In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd
from collections import defaultdict



def pairwise_distance(x):
    with torch.no_grad():
        x_inner = -2 * torch.matmul(x, x.transpose(1, 2))
        x_square = torch.sum(x ** 2, dim=-1, keepdim=True)
        return x_square + x_inner + x_square.transpose(1, 2)

def knn_graph(x, k):
    dist = pairwise_distance(x)
    _, idx = torch.topk(-dist, k=k, dim=-1)
    B, N, _ = idx.shape
    center = torch.arange(N, device=x.device).view(1, N, 1).expand(B, N, k)
    return torch.stack([idx, center], dim=0)

def get_spatial_knn_edge_index(grid_size, k, device):
    positions = []
    for i in range(grid_size):
        for j in range(grid_size):
            x = i / (grid_size - 1) if grid_size > 1 else 0.5
            y = j / (grid_size - 1) if grid_size > 1 else 0.5
            positions.append([x, y])
    positions = torch.tensor(positions, dtype=torch.float32, device=device)

    N = positions.shape[0]
    dist = torch.cdist(positions, positions, p=2)
    _, idx = torch.topk(-dist, k=k+1, dim=-1)
    idx = idx[:, 1:]
    center = torch.arange(N, device=device).view(1, N, 1).expand(1, N, k)
    edge_index = torch.stack([idx.unsqueeze(0), center], dim=0)
    return edge_index.squeeze(1)

class MRConv(nn.Module):
    def __init__(self, in_channels, out_channels, bias=True):
        super().__init__()
        self.linear = nn.Linear(in_channels * 2, out_channels, bias=bias)

    def forward(self, x, edge_index):
        B, N, C = x.shape
        if edge_index.dim() == 3:
            edge_index = edge_index.unsqueeze(1).expand(-1, B, -1, -1)

        k = edge_index.shape[-1]
        src_idx = edge_index[0]
        tgt_idx = edge_index[1]

        batch_idx = torch.arange(B, device=x.device).view(B, 1, 1).expand(B, N, k)
        x_src = x[batch_idx, src_idx]
        x_tgt = x[batch_idx, tgt_idx]


        x_center = x.unsqueeze(2)
        x_diff = x_src - x_center
        x_agg, _ = torch.max(x_diff, dim=2)

        x_concat = torch.cat([x_center.squeeze(2), x_agg], dim=-1)
        return self.linear(x_concat)

class Grapher(nn.Module):

    def __init__(self, dim, k=9, expansion_ratio=2, dropout=0.0, spatial_knn=False,
                 grid_size=4, device=None):
        super().__init__()
        self.k = k
        self.spatial_knn = spatial_knn
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim, bias=False)
        self.bn1 = nn.BatchNorm1d(dim)
        self.graph_conv = MRConv(dim, dim * expansion_ratio, bias=True)
        self.fc2 = nn.Linear(dim * expansion_ratio, dim, bias=False)
        self.bn2 = nn.BatchNorm1d(dim)
        self.dropout = nn.Dropout(dropout)

        if spatial_knn:

            self.register_buffer('spatial_edge_index',
                                 get_spatial_knn_edge_index(grid_size, k, device))

    def forward(self, x):

        identity = x
        x = self.norm(x)
        x = self.fc1(x)
        x = self.bn1(x.transpose(1,2)).transpose(1,2)
        x = F.gelu(x)

        if self.spatial_knn:

            edge_index = self.spatial_edge_index
        else:

            edge_index = knn_graph(x, self.k)

        x = self.graph_conv(x, edge_index)
        x = self.fc2(x)
        x = self.bn2(x.transpose(1,2)).transpose(1,2)
        x = self.dropout(x)
        return x + identity

class FFN(nn.Module):
    def __init__(self, dim, expansion_ratio=4, dropout=0.0):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim, dim * expansion_ratio)
        self.fc2 = nn.Linear(dim * expansion_ratio, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        identity = x
        x = self.norm(x)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x + identity

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=8, patch_size=2, in_channels=2, embed_dim=64):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = x.permute(0, 3, 1, 2)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class VisionGNN(nn.Module):
    def __init__(self, img_size=8, patch_size=2, in_channels=2, num_classes=9,
                 embed_dim=64, depth=6, k=9, expansion_ratio=2, ffn_expansion=4,
                 dropout=0.1, spatial_knn=False):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        grid_size = img_size // patch_size

        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                'grapher': Grapher(embed_dim, k=k, expansion_ratio=expansion_ratio,
                                   dropout=dropout, spatial_knn=spatial_knn,
                                   grid_size=grid_size, device=None),
                'ffn': FFN(embed_dim, expansion_ratio=ffn_expansion, dropout=dropout)
            }) for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        for blk in self.blocks:
            x = blk['grapher'](x)
            x = blk['ffn'](x)

        x = x.mean(dim=1)
        x = self.norm(x)
        return self.head(x)

# -------------------------------
# 2. Dataset (unchanged)
# -------------------------------

def load_dataset():
    X = np.load('/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/otfs_amc_data.npy')
    y = np.load('/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/otfs_amc_labels.npy')
    snr = np.load('/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/otfs_amc_snr.npy')
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    snr_tensor = torch.tensor(snr, dtype=torch.float32)
    return X_tensor, y_tensor, snr_tensor

class SNRDataset(TensorDataset):
    def __init__(self, X, y, snr):
        super().__init__(X, y)
        self.snr = snr

    def __getitem__(self, idx):
        return (self.tensors[0][idx], self.tensors[1][idx], self.snr[idx])

def create_data_loaders(X, y, snr, test_size=0.2, val_size=0.1, batch_size=256):
    dataset = SNRDataset(X, y, snr)
    total = len(dataset)
    test_len = int(total * test_size)
    val_len = int(total * val_size)
    train_len = total - test_len - val_len
    train_data, val_data, test_data = random_split(
        dataset, [train_len, val_len, test_len], generator=torch.Generator().manual_seed(42)
    )

    def collate_fn(batch):
        inputs = torch.stack([item[0] for item in batch])
        labels = torch.stack([item[1] for item in batch])
        snrs = torch.stack([item[2] for item in batch])
        return inputs, labels, snrs

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_data, batch_size=batch_size, collate_fn=collate_fn)
    test_loader = DataLoader(test_data, batch_size=batch_size, collate_fn=collate_fn)
    return train_loader, val_loader, test_loader

X, y, snr = load_dataset()
mod_schemes = ['BPSK', 'QPSK', '8PSK', '8QAM', '16QAM', '32QAM', '64QAM', '128QAM', '256QAM']
train_loader, val_loader, test_loader = create_data_loaders(X, y, snr)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = VisionGNN(
    img_size=32,
    patch_size=2,
    in_channels=2,
    num_classes=len(mod_schemes),
    embed_dim=64,
    depth=6,
    k=12,
    expansion_ratio=2,
    ffn_expansion=4,
    dropout=0.1,
    spatial_knn=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
max_grad_norm = 1.0


def train_model(model, train_loader, val_loader, epochs=50):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels, _ in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            running_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            torch.cuda.empty_cache()
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels, _ in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, pred = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (pred == labels).sum().item()
        val_loss = val_loss / len(val_loader)
        val_acc = correct / total
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    return train_losses, train_accs, val_losses, val_accs

train_losses, train_accs, val_losses, val_accs = train_model(model, train_loader, val_loader, epochs=30)


def evaluate_model(model, test_loader):
    model.eval()
    all_preds, all_labels, all_snrs = [], [], []
    with torch.no_grad():
        for inputs, labels, snrs in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, pred = torch.max(outputs, 1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_snrs.extend(snrs.cpu().numpy())
    return all_preds, all_labels, all_snrs

all_preds, all_labels, all_snrs = evaluate_model(model, test_loader)
overall_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"\nOverall Test Accuracy: {overall_acc:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=mod_schemes))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=mod_schemes, yticklabels=mod_schemes)
plt.title(f'Overall Confusion Matrix (Acc: {overall_acc:.4f})')
plt.ylabel('True')
plt.xlabel('Pred')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/overall_cm_spatial.png')
plt.show()

snr_vals = sorted(set(all_snrs))
snr_acc = []
for snr_val in snr_vals:
    idx = [i for i, s in enumerate(all_snrs) if s == snr_val]
    if idx:
        acc = np.mean(np.array(all_preds)[idx] == np.array(all_labels)[idx])
        snr_acc.append(acc)

plt.figure()
plt.plot(snr_vals, snr_acc, 'o-')
plt.xlabel('SNR (dB)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs SNR (Spatial KNN)')
plt.grid(True)
plt.savefig('/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/acc_vs_snr_spatial.png')
plt.show()

torch.save(model.state_dict(), '/content/drive/MyDrive/AMC_VGNN/32/full_otfs_amc_dataset_32_32_300/otfs_amc_vig_spatial.pth')
print("Model saved as 'otfs_amc_vig_spatial.pth'")